# OpenAI framework example (compare offers)

Follow up the readme.md to get started

In [73]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, ModelSettings
from agents import set_default_openai_client, set_default_openai_api, set_tracing_disabled
from openai import OpenAI, AsyncOpenAI
from agents.decorators import input_guardrail
from agents import Runner, GuardrailFunctionOutput
from agents import InputGuardrailTripwireTriggered  
from pydantic import BaseModel, Field
import os
import asyncio

In [74]:
#offer_names = ["prosermo", "enpal", "arndt", "widmann"]
offer_names = ["prosermo", "arndt"]

In [75]:
# Generator function. Yields result to calling function
def handle_files_generator(files):
    for filename in files:
        # This is not printing the result
        yield filename + ".txt"

def read_offer_file(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        print(f"File not found: {filename}")
        return "No content available"

# Call generator as an iterator and print the result
for file in handle_files_generator(offer_names):
    print(file)

prosermo.txt
arndt.txt


In [76]:
class EnergyOffer(BaseModel):
    concerns_PV: bool = Field(description="Whether the offer is related to photovoltaics (PV) and wallbox")
    concerns_WP: bool = Field(description="Whether the offer is related to heat pumps (WP)")
    PV_offer: str = Field(description="The photovoltaic (PV) and wallbox related offer details formatted as markdown, starting with the supplier name")
    WP_offer: str = Field(description="The heat pump (WP) related offer details formatted as markdown, starting with the supplier name")

    def to_markdown(self) -> str:
        # Custom method to convert the model to a Markdown representation
        return (
            f"# PV offer: \n{self.PV_offer}\n\n"
            f"# WP offer: \n{self.WP_offer}\n\n"
        )

class OfferCompare(BaseModel):
    PV_ranking: str = Field(description="How the PV offers are ranked compared to the other offers (1 = best, 2 = second best, etc.) and why")
    WP_ranking: str = Field(description="How the WP offers are ranked compared to the other offers (1 = best, 2 = second best, etc.) and why")
    Proposal: str = Field(description="Which combination of PV and WP offers is proposed as the best option and why")
    
class OfferGuardrail(BaseModel):
    review: str = Field(description="The review of the offer")
    tripwire_triggered: bool = Field(description="Whether the tripwire was triggered based on the review")

#EnergyOffer.model_json_schema()

In [77]:
load_dotenv(override=True)

MODEL_NAME = os.getenv("MODEL_NAME", "gpt-5.4-mini")
BASE_URL = os.getenv("BASE_URL", "https://api.openai.com/v1")
API_KEY = os.getenv("API_KEY")

#print("API key loaded:", bool(API_KEY))
#print("Base URL:", BASE_URL)
#print("Model Name:", MODEL_NAME)

AI_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
# The Agents SDK ignores base_url/api_key in ModelSettings: register the client globally
set_default_openai_client(AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY))
set_default_openai_api("chat_completions")
set_tracing_disabled(True)
settings = ModelSettings()

In [78]:
guardrail_checker = Agent(name="checker", 
                instructions="You review if the offer contains technical information about photovoltaics (PV) or heat pumps (WP) and trigger the tripwire if it does not", 
                model=MODEL_NAME, model_settings=settings, output_type=OfferGuardrail)

@input_guardrail(run_in_parallel=False)
async def offer_guardrail(ctx, agent, message):
    result = await Runner.run(guardrail_checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.tripwire_triggered
    print(is_problem, review)
    return GuardrailFunctionOutput(output_info={"review": review}, tripwire_triggered=is_problem)

## Check with guardrail if offer is technical and topic related, then route offers to PV or WP

In [79]:
instructions = "You route the offers depending on their content (PV or WP)."
content_router_agent = Agent(name="content_router", instructions=instructions, output_type=EnergyOffer, 
                             model=MODEL_NAME, model_settings=settings, input_guardrails=[offer_guardrail])

tasks = []

async def route_offers(offer_name):
    content = read_offer_file("../docs/" + offer_name)
    try:
        result = await Runner.run(content_router_agent, offer_name + "\n" + content)
        return result
    except InputGuardrailTripwireTriggered as e:
        print(f"Tripwire triggered for offer {offer_name}: {e}")
        return None

async def run_searches():
    tasks = [route_offers(offer_name) for offer_name in handle_files_generator(offer_names)]
    results = await asyncio.gather(*tasks)
    return results

with trace("Routing offers based on content"):
    results = await run_searches()
    print(results)

reports = ""
offer_results = [result.final_output.to_markdown() for result in results if result is not None]
for offer in offer_results:
    print(offer)
    reports += offer + "\n\n"

True review='The offer is a fictional travel sample and contains no technical information about photovoltaics (PV) or heat pumps (WP).' tripwire_triggered=True
Tripwire triggered for offer prosermo.txt: Guardrail InputGuardrail triggered tripwire
False review='The text contains clear technical photovoltaic content, including PV module wattage, system size in kWp, hybrid inverter, battery storage capacity, installation components, commissioning, and related services. It is explicitly about a photovoltaic system offer, so it qualifies as technical PV information.' tripwire_triggered=False
[None, RunResult(input='arndt.txt\nSure. To keep it safe for demos, testing, training, or comparison purposes, I’d make it clearly fictional and marked “SAMPLE / NOT A REAL OFFER.”\n\nBelow are three usable variants based on a typical residential PV installation in Germany.\n\nOption 1: Realistic professional offer\n\nSAMPLE / NOT A REAL COMMERCIAL OFFER\n\nSolarNova Muster GmbH\n Example Street 42\n 70

In [80]:
#reports = ""
# Alternative sequential approach
if False:
    with trace("Routing offers based on content"):
        for offer_name in handle_files_generator(offer_names):
            result = await Runner.run(content_router_agent, read_offer_file("../docs/" + offer_name))
            reports = reports + "\n" + f"\nOffer: {offer_name}\n\n{result.final_output.to_markdown()}"
            #print(f"Offer: {offer_name}")
            #print(f"Routing Result: PV is {result.final_output.concerns_PV}, WP is {result.final_output.concerns_WP}")
            #print(f"PV Offer Details: {result.final_output.PV_offer}")
            #print(f"WP Offer Details: {result.final_output.WP_offer}")
    print(reports)

In [81]:
instructions = "You compare the PV and WP offers from different suppliers based on technical, quality and economical aspects. Format your response as markdown."
compare_agent = Agent(name="compare_agent", instructions=instructions, output_type=OfferCompare, model=MODEL_NAME, model_settings=settings)

result = await Runner.run(compare_agent, reports)
print(result.final_output)
print(result)
with open("../docs/comparison_report.md", "w", encoding="utf-8") as f:
    f.write(f"# PV ranking\n\n{result.final_output.PV_ranking}\n\n")
    f.write(f"# WP ranking\n\n{result.final_output.WP_ranking}\n\n")
    f.write(f"# Proposal\n\n{result.final_output.Proposal}\n")



PV_ranking='1. SolarNova Muster GmbH ranks best among the PV offers because it provides a complete, technically coherent system: 10.8 kWp PV array, 10.2 kWh battery, 10 kW hybrid inverter, installation, commissioning, and documentation are all included. From a quality perspective, the offer is well structured and comprehensive, with no obvious gaps in scope. Economically, the total price of €19,950 is plausible for this system size and storage capacity, so it appears competitive for a turnkey package.' WP_ranking='No WP offer was provided, so a ranking is not possible. There is no basis to compare technical quality, scope of supply, or price against other heat pump offers.' Proposal='For the PV side, the best available option is SolarNova Muster GmbH because it is the only PV offer provided and it is technically complete and economically reasonable. For WP, no proposal can be made because no WP offer was submitted. Therefore, the best overall combination cannot be fully determined from